Create Tables

In [ ]:
import sqlite3

conn = sqlite3.connect('ao3_works.db')
c = conn.cursor()
c.execute('''
    CREATE TABLE IF NOT EXISTS all_works (
    id TEXT PRIMARY KEY,
    published TEXT,
    link TEXT,
    status TEXT,
    title TEXT,
    author TEXT,
    rating TEXT,
    categories TEXT,
    fandoms TEXT,
    relationships TEXT,
    characters TEXT,
    freeform_tags TEXT,
    summary TEXT
    )
''')

conn.commit()
conn.close()


In [ ]:
import sqlite3

conn = sqlite3.connect('ao3_works.db')
c = conn.cursor()

c.execute('''
    CREATE TABLE IF NOT EXISTS my_bookmarks (
        work_id TEXT PRIMARY KEY
    )
''')

conn.commit()

c.execute('''
    CREATE TABLE IF NOT EXISTS public_bookmarks (
        user_id TEXT,
        work_id TEXT,
        PRIMARY KEY (user_id, work_id)
    )
''')

conn.commit()
conn.close()

print("SQLite table 'my_bookmarks' and 'public_bookmarks' created successfully in ao3_works.db")

SQLite table 'my_bookmarks' and 'public_bookmarks' created successfully in ao3_works.db


Scrape Pages and Insert Data to the DB

In [ ]:
import time
import re
import math
from collections import Counter
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

import sqlite3

In [ ]:
BASE_URL = "https://archiveofourown.org"

HEADERS = {
    "User-Agent": "AO3 recommender student project - small-scale metadata parser"
}

In [ ]:
import time
import requests

def get_html(
    url: str,
    delay: float = 2.0,
    retries: int = 3,
    backoff: float = 2.0
) -> str:
    """
    Fetch a page slowly and politely, retrying temporary failures.
    """
    last_error = None

    for attempt in range(1, retries + 1):
        time.sleep(delay)

        try:
            response = requests.get(
                url,
                headers=HEADERS,
                timeout=(10, 20)  # connect timeout, read timeout
            )

            if response.status_code == 200:
                print(f"Fetch successful for {url}.")
                return response.text

            last_error = Exception(
                f"Failed to fetch {url}. Status code: {response.status_code}"
            )

        except requests.exceptions.Timeout as e:
            last_error = e

        except requests.exceptions.ConnectionError as e:
            last_error = e

        if attempt < retries:
            wait = backoff * attempt
            print(f"Fetch failed for {url}. Retrying in {wait:.1f}s...")
            time.sleep(wait)

    raise Exception(f"Failed to fetch {url} after {retries} attempts: {last_error}")

In [ ]:
def extract_work_id(url: str) -> str | None:
    """
    Returns a work ID string from URLs like:
    https://archiveofourown.org/works/33023536
    """
    match = re.search(r"/works/(\d+)", url)

    if match:
        return match.group(1)

    return None

In [ ]:
def clean_number(value: str):
    """
    Returns a string from '104,439' into 104439.
    """
    if not value:
        return None

    value = value.replace(",", "").strip()

    if value.isdigit():
        return int(value)

    return value

In [ ]:
def extract_tag_texts(soup: BeautifulSoup, dd_class: str) -> list[str]:
    """
    Returns tag text from a dd section, e.g.
    dd.rating, dd.fandom, dd.relationship, dd.character, dd.freeform.
    """
    section = soup.select_one(f"dd.{dd_class}")

    if not section:
        return []

    return [tag.get_text(strip=True) for tag in section.select("a.tag")]

In [ ]:
def extract_stats_text(soup: BeautifulSoup) -> dict:
    """
    Parses the stats section:
    Published, Updated, Words, Chapters, Comments, Kudos, Bookmarks, Hits.
    """
    stats = {}

    stats_dl = soup.select_one("dl.stats")

    if not stats_dl:
        return stats

    current_key = None

    for element in stats_dl.find_all(["dt", "dd"], recursive=False):
        if element.name == "dt":
            current_key = element.get_text(strip=True).replace(":", "").lower()
        elif element.name == "dd" and current_key:
            stats[current_key] = clean_number(element.get_text(strip=True))
            current_key = None

    return stats

In [ ]:
import re
def parse_work_page(html: str, url: str) -> dict:
    """
    Parses one AO3 work page and returns metadata.
    """
    soup = BeautifulSoup(html, "html.parser")

    title_element = soup.select_one("h2.title")
    author_element = soup.select_one("h3.byline")

    rating = extract_tag_texts(soup, "rating")
    # warnings = extract_tag_texts(soup, "warning")
    categories = extract_tag_texts(soup, "category")
    fandoms = extract_tag_texts(soup, "fandom")
    relationships = extract_tag_texts(soup, "relationship")
    characters = extract_tag_texts(soup, "character")
    freeform_tags = extract_tag_texts(soup, "freeform")

    language_element = soup.select_one("dd.language")
    summary_element = soup.select_one("div.summary blockquote")

    stats = extract_stats_text(soup)

    work_id = extract_work_id(url)

    for relationship in relationships:
      split_relationship = re.split(r'[/|&]', relationship)
      # split_relationship = relationship.split('/')
      # print(split_relationship)
      for character in split_relationship:
        if character not in characters:
          characters.append(character)

    work_data = {
        "work_id": work_id,
        "url": url,
        "title": title_element.get_text(" ", strip=True) if title_element else None,
        "author": author_element.get_text(" ", strip=True) if author_element else None,
        "rating": rating,
        # "warnings": warnings,
        "categories": categories,
        "fandoms": fandoms,
        "relationships": relationships,
        "characters": characters,
        "freeform_tags": freeform_tags,
        "language": language_element.get_text(strip=True) if language_element else None,
        "summary": summary_element.get_text(" ", strip=True) if summary_element else None,
        "published": stats.get("published"),
        # "updated": stats.get("updated") or stats.get("status"),
        "words": stats.get("words"),
        "chapters": stats.get("chapters"),
        "comments": stats.get("comments"),
        "kudos": stats.get("kudos"),
        "bookmarks_count": stats.get("bookmarks"),
        "hits": stats.get("hits"),
    }

    return work_data

In [ ]:
def get_total_bookmark_pages(bookmarks_count, per_page=20):
    """
    Returns an int of the estimated number of bookmark user pages.
    AO3 bookmark pages usually show 20 bookmarks per page.
    Example: 832 bookmarks -> ceil(832 / 20) = 42 pages.
    """
    if not bookmarks_count:
        return 0

    try:
        bookmarks_count = int(bookmarks_count)
    except ValueError:
        return 0

    return math.ceil(bookmarks_count / per_page)

In [ ]:
def get_bookmarks_page_url(work_url: str, page: int) -> str:
    """
    Returns a work URL with a bookmarks page number.
    Example:
    https://archiveofourown.org/works/33023536
    ->
    https://archiveofourown.org/works/33023536/bookmarks?page=1
    """
    work_id = extract_work_id(work_url)

    if not work_id:
        raise ValueError(f"Could not extract work ID from {work_url}")

    return f"{BASE_URL}/works/{work_id}/bookmarks?page={page}"


In [ ]:


def parse_bookmark_users(html: str) -> list[str]:
    soup = BeautifulSoup(html, "html.parser")
    users = set()

    for link in soup.select('a[href*="/pseuds/"]'):
        href = link.get("href", "")

        match = re.match(r"^/users/([^/]+)/pseuds/([^/?#]+)", href)

        if match:
            users.add(match.group(2))

    return list(users)

In [ ]:
def collect_all_bookmark_users(work_url: str, bookmarks_count: int, delay: float = 3.0) -> list[str]:
    """
    Returns all users that bookmarked a work.
    """
    total_pages = get_total_bookmark_pages(bookmarks_count)

    # print(f"Bookmark count: {bookmarks_count}")
    # print(f"Estimated bookmark pages: {total_pages}")

    all_users = set()

    for page in range(1, total_pages + 1):
        bookmarks_url = get_bookmarks_page_url(work_url, page)
        print(f"Fetching bookmarks page {page}/{total_pages}: {bookmarks_url}")

        try:
            bookmarks_html = get_html(bookmarks_url, delay=delay)
            page_users = parse_bookmark_users(bookmarks_html)
            all_users.update(page_users)

        except Exception as e:
            print(f"Could not fetch bookmark page {page} for {work_url}: {e}")

    return list(all_users)


In [ ]:
def update_my_interactions_table(c, work_id):
    c.execute(
        "INSERT OR IGNORE INTO my_bookmarks (work_id) VALUES (?)",
        (work_id,)
    )


def update_public_interactions_table(c, user_id, work_id):
    c.execute(
        "INSERT OR IGNORE INTO public_bookmarks (user_id, work_id) VALUES (?, ?)",
        (user_id, work_id)
    )

In [ ]:
def update_all_works_table_status(c, work_id, new_status):
    c.execute(
        "UPDATE all_works SET status = ? WHERE id = ?",
        (new_status, work_id)
    )
    print(f"Work {work_id} status updated to {new_status}.")

In [ ]:


def update_all_works_table_tags(
    c, work_id, rating, categories, fandoms,
    relationships, characters, freeform_tags, summary
):
    c.execute(
        '''UPDATE all_works
           SET
               rating = ?,
               categories = ?,
               fandoms = ?,
               relationships = ?,
               characters = ?,
               freeform_tags = ?,
               summary = ?
           WHERE id = ?''',
        (
            rating, categories, fandoms, relationships,
            characters, freeform_tags, summary, work_id
        )
    )
    print(f"Work {work_id} updated with tags.")

In [ ]:
def collect_work_data_and_bookmark_usernames_from_a_link(work_url: str) -> tuple[dict, list[str]]:
    """
    Main function:
    - Takes AO3 work URLs
    - Parses work metadata
    - Parses public bookmark users
    """
    # works = []
    # bookmark_rows = []

    print(f"Fetching work: {work_url}")

    try:
        work_html = get_html(work_url)
    except requests.exceptions.ReadTimeout as e:
        print(f"Read timeout fetching work {work_url}: {e}. Skipping this work.")
        return None, []
    except Exception as e:
        print(f"Error fetching work {work_url}: {e}. Skipping this work.")
        return None, []

    work_data = parse_work_page(work_html, work_url)

    # Portfolio-safe default: skip Explicit works
    ratings = work_data.get("rating", [])
    if "Explicit" in ratings:
        print(f"Skipping explicit work: {work_url}")
        return None, []

    # works.append(work_data)
    bookmarks_count = work_data.get("bookmarks_count", 0)

    try:
        bookmark_users = collect_all_bookmark_users(
            work_url=work_url,
            bookmarks_count=bookmarks_count,
            delay=3.0
        )
        if len(bookmark_users) == 0:
            bookmark_users = [work_data.get('author')]

    except Exception as e:
        print(f"Could not fetch bookmarks for {work_url}: {e}")
        bookmark_users = [work_data.get('author')]

    return work_data, bookmark_users

In [ ]:
import json

def collect_and_update_with_new_works(links_list, update_my_interactions):
    conn = sqlite3.connect('ao3_works.db')
    c = conn.cursor()

    try:
        for link in links_list:
            work_data, bookmark_users = collect_work_data_and_bookmark_usernames_from_a_link(link)

            if work_data is None:
                # add to another table [link, rss/mine], so it can be tried again
                continue

            work_id = work_data.get('work_id')

            try:
                if update_my_interactions:
                    add_work_to_db(
                        c,
                        work_id,
                        work_data.get('published'),
                        work_data.get('url'),
                        'New',
                        work_data.get('title'),
                        work_data.get('author')
                    )
                    update_my_interactions_table(c, work_id)

                for username in bookmark_users:
                    update_public_interactions_table(c, username, work_id)

                update_all_works_table_tags(
                    c,
                    work_id,
                    json.dumps(work_data.get('rating')),
                    json.dumps(work_data.get('categories')),
                    json.dumps(work_data.get('fandoms')),
                    json.dumps(work_data.get('relationships')),
                    json.dumps(work_data.get('characters')),
                    json.dumps(work_data.get('freeform_tags')),
                    work_data.get('summary')
                )

                update_all_works_table_status(
                    c,
                    work_id,
                    'Scraped'
                )

                conn.commit()

            except Exception as e:
                conn.rollback()
                print(f"Error adding work {work_id}: {e}")

    finally:
        conn.close()



In [ ]:
if __name__ == "__main__":
    ao3_links = [
        # Put your AO3 work links here
        # "https://archiveofourown.org/works/83067686",
        # "https://archiveofourown.org/works/33023536",
        "https://archiveofourown.org/works/75267501",
        'https://archiveofourown.org/works/56181721',
        'https://archiveofourown.org/works/81993896',
        'https://archiveofourown.org/works/76423276',
        'https://archiveofourown.org/works/55165036',
        'https://archiveofourown.org/works/50992918',
        'https://archiveofourown.org/works/73894251',
        'https://archiveofourown.org/works/62910172'

    ]

    collect_and_update_with_new_works(ao3_links,True)

Fetching work: https://archiveofourown.org/works/75267501
Fetch successful for https://archiveofourown.org/works/75267501.
Fetching bookmarks page 1/8: https://archiveofourown.org/works/75267501/bookmarks?page=1
Fetch successful for https://archiveofourown.org/works/75267501/bookmarks?page=1.
Fetching bookmarks page 2/8: https://archiveofourown.org/works/75267501/bookmarks?page=2
Fetch failed for https://archiveofourown.org/works/75267501/bookmarks?page=2. Retrying in 2.0s...
Fetch successful for https://archiveofourown.org/works/75267501/bookmarks?page=2.
Fetching bookmarks page 3/8: https://archiveofourown.org/works/75267501/bookmarks?page=3
Fetch successful for https://archiveofourown.org/works/75267501/bookmarks?page=3.
Fetching bookmarks page 4/8: https://archiveofourown.org/works/75267501/bookmarks?page=4
Fetch successful for https://archiveofourown.org/works/75267501/bookmarks?page=4.
Fetching bookmarks page 5/8: https://archiveofourown.org/works/75267501/bookmarks?page=5
Fetch 

In [ ]:
conn = sqlite3.connect('ao3_works.db')
c = conn.cursor()

all = c.execute('''
      Select * from all_works
''')

for row in all:
    print(row)

conn.close()

('75267501', '2025-12-21', 'https://archiveofourown.org/works/75267501', 'Scraped', 'the wolf and the fox', 'GoddessOfWriting , TheDauntless (GoddessOfWriting)', '["Mature"]', '["F/M"]', '["A Song of Ice and Fire - George R. R. Martin", "A Song of Ice and Fire & Related Fandoms", "Game of Thrones (TV)"]', '["Jon Snow/Original Female Character(s)", "Jon Snow & Original Female Character(s)", "Jon Snow & Arya Stark & Bran Stark & Rickon Stark & Robb Stark & Sansa Stark", "Ghost | Jon Snow\'s Direwolf & Jon Snow", "Catelyn Tully Stark/Ned Stark", "Jon Snow & Ned Stark"]', '["Original Female Character(s)", "Characters From The North (A Song of Ice and Fire)", "Jon Snow", "Ghost | Jon Snow\'s Direwolf", "Arya Stark", "Nymeria | Arya Stark\'s Direwolf", "Sansa Stark", "Robb Stark", "Theon Greyjoy", "Bran Stark", "Rickon Stark", "Summer | Bran Stark\'s Direwolf", "Shaggydog | Rickon Stark\'s Direwolf", "Ned Stark", "Catelyn Tully Stark", "Maester Luwin of Winterfell (A Song of Ice and Fire)", 

In [ ]:
conn = sqlite3.connect('ao3_works.db')
c = conn.cursor()

all = c.execute('''
      Select * from my_bookmarks
''')

for row in all:
    print(row)

conn.close()

('75267501',)
('56181721',)
('81993896',)
('76423276',)
('55165036',)
('50992918',)
('73894251',)
('62910172',)


In [ ]:
conn = sqlite3.connect('ao3_works.db')
c = conn.cursor()

all = c.execute('''
      Select * from public_bookmarks
''')

for row in all:
    print(row)

conn.close()

RSS feed to all works sql

In [ ]:
!pip install feedparser==6.0.12
import feedparser

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.7 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=eaba131cd6b06ed272357a28c8add425f42860506280d195a1bdd3297cb1c698
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [ ]:
# import sqlite3
# def add_work_to_db(c, work_id, published, link, status, title, author):
#     try:
#         c.execute('''
#             INSERT INTO all_works (id, published, link, status, title, author)
#             VALUES (?, ?, ?, ?, ?, ?)
#         ''', (work_id, published, link, status, title, author))
#         print(f"Successfully added work {work_id} to the database.")
#     except sqlite3.IntegrityError:
#         print(f"Work {work_id} already exists in the database. Skipping.")
#     except Exception as e:
#         print(f"Error adding work {work_id}: {e}")
def add_work_to_db(c, work_id, published, link, status, title, author):
    c.execute('''
        INSERT OR IGNORE INTO all_works
        (id, published, link, status, title, author)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (work_id, published, link, status, title, author))

    if c.rowcount == 0:
        print(f"Work {work_id} already exists in the database. Skipping.")
    else:
        print(f"Successfully added work {work_id} to the database.")


Get new works from RSS feeds to add the DB

In [ ]:
import feedparser
feed_url = "https://archiveofourown.org/tags/2007008/feed.atom"

feed = feedparser.parse(feed_url)

conn = sqlite3.connect('ao3_works.db')
c = conn.cursor()


for entry in feed.entries:
    work_id = extract_work_id(entry.get("link"))
    published = entry.get("published")
    link = entry.get("link")
    title = entry.get("title")
    author = entry.get("author")
    status = "New"

    if work_id:
        add_work_to_db(c, work_id, published, link, status, title, author)
    else:
        print(f"Could not extract work ID for entry: {link}")
conn.commit()
conn.close()

Successfully added work 84825481 to the database.
Successfully added work 84823306 to the database.
Successfully added work 84821151 to the database.
Successfully added work 84819631 to the database.
Successfully added work 84819371 to the database.
Successfully added work 84818236 to the database.
Successfully added work 84817521 to the database.
Successfully added work 84817486 to the database.
Successfully added work 84816756 to the database.
Successfully added work 84814161 to the database.
Successfully added work 84810946 to the database.
Successfully added work 84810556 to the database.
Successfully added work 84808636 to the database.
Successfully added work 84807446 to the database.
Successfully added work 84805866 to the database.
Successfully added work 84805206 to the database.
Successfully added work 84804186 to the database.
Successfully added work 84803571 to the database.
Successfully added work 84802621 to the database.
Successfully added work 84802421 to the database.


In [ ]:
import sqlite3
conn = sqlite3.connect('ao3_works.db')
c = conn.cursor()

all = c.execute('''
      Select * from all_works
''')

for row in all:
    print(row)

conn.close()

('75267501', '2025-12-21', 'https://archiveofourown.org/works/75267501', 'Scraped', 'the wolf and the fox', 'GoddessOfWriting , TheDauntless (GoddessOfWriting)', '["Mature"]', '["F/M"]', '["A Song of Ice and Fire - George R. R. Martin", "A Song of Ice and Fire & Related Fandoms", "Game of Thrones (TV)"]', '["Jon Snow/Original Female Character(s)", "Jon Snow & Original Female Character(s)", "Jon Snow & Arya Stark & Bran Stark & Rickon Stark & Robb Stark & Sansa Stark", "Ghost | Jon Snow\'s Direwolf & Jon Snow", "Catelyn Tully Stark/Ned Stark", "Jon Snow & Ned Stark"]', '["Original Female Character(s)", "Characters From The North (A Song of Ice and Fire)", "Jon Snow", "Ghost | Jon Snow\'s Direwolf", "Arya Stark", "Nymeria | Arya Stark\'s Direwolf", "Sansa Stark", "Robb Stark", "Theon Greyjoy", "Bran Stark", "Rickon Stark", "Summer | Bran Stark\'s Direwolf", "Shaggydog | Rickon Stark\'s Direwolf", "Ned Stark", "Catelyn Tully Stark", "Maester Luwin of Winterfell (A Song of Ice and Fire)", 

Get 'new' works from 'all_works' that are older than k days to be scraped and enrich the DB

In [ ]:
conn = sqlite3.connect('ao3_works.db')
c = conn.cursor()



overdue_link_rows = c.execute('''
      Select link from all_works
      where status = 'New'
''').fetchall()

overdue_link_list = [row[0] for row in overdue_link_rows]

conn.close()

In [ ]:
overdue_link_list

['https://archiveofourown.org/works/84723811',
 'https://archiveofourown.org/works/84723496',
 'https://archiveofourown.org/works/84723001',
 'https://archiveofourown.org/works/84722136',
 'https://archiveofourown.org/works/84721256',
 'https://archiveofourown.org/works/84720706',
 'https://archiveofourown.org/works/84719016',
 'https://archiveofourown.org/works/84715966',
 'https://archiveofourown.org/works/84715021',
 'https://archiveofourown.org/works/84714196',
 'https://archiveofourown.org/works/84714186',
 'https://archiveofourown.org/works/84711981',
 'https://archiveofourown.org/works/84711126',
 'https://archiveofourown.org/works/84711101',
 'https://archiveofourown.org/works/84709216',
 'https://archiveofourown.org/works/84708881',
 'https://archiveofourown.org/works/84708701',
 'https://archiveofourown.org/works/84708291',
 'https://archiveofourown.org/works/84705176',
 'https://archiveofourown.org/works/84702331',
 'https://archiveofourown.org/works/84702261',
 'https://arc

In [ ]:
for link in overdue_link_list:
    print(link)

https://archiveofourown.org/works/84723811
https://archiveofourown.org/works/84723496
https://archiveofourown.org/works/84723001
https://archiveofourown.org/works/84722136
https://archiveofourown.org/works/84721256
https://archiveofourown.org/works/84720706
https://archiveofourown.org/works/84719016
https://archiveofourown.org/works/84715966
https://archiveofourown.org/works/84715021
https://archiveofourown.org/works/84714196
https://archiveofourown.org/works/84714186
https://archiveofourown.org/works/84711981
https://archiveofourown.org/works/84711126
https://archiveofourown.org/works/84711101
https://archiveofourown.org/works/84709216
https://archiveofourown.org/works/84708881
https://archiveofourown.org/works/84708701
https://archiveofourown.org/works/84708291
https://archiveofourown.org/works/84705176
https://archiveofourown.org/works/84702331
https://archiveofourown.org/works/84702261
https://archiveofourown.org/works/84701301
https://archiveofourown.org/works/84699031
https://arc

In [ ]:
collect_and_update_with_new_works(overdue_link_list, False)

Fetching work: https://archiveofourown.org/works/84723811
Fetch failed for https://archiveofourown.org/works/84723811. Retrying in 2.0s...
Fetch successful for https://archiveofourown.org/works/84723811.
Skipping explicit work: https://archiveofourown.org/works/84723811
Fetching work: https://archiveofourown.org/works/84723496
Fetch successful for https://archiveofourown.org/works/84723496.
Work 84723496 updated with tags.
Work 84723496 status updated to Scraped.
Fetching work: https://archiveofourown.org/works/84723001
Fetch failed for https://archiveofourown.org/works/84723001. Retrying in 2.0s...
Fetch successful for https://archiveofourown.org/works/84723001.
Work 84723001 updated with tags.
Work 84723001 status updated to Scraped.
Fetching work: https://archiveofourown.org/works/84722136
Fetch successful for https://archiveofourown.org/works/84722136.
Work 84722136 updated with tags.
Work 84722136 status updated to Scraped.
Fetching work: https://archiveofourown.org/works/84721256

In [ ]:
conn = sqlite3.connect('ao3_works.db')
c = conn.cursor()

all = c.execute('''
      Select * from all_works
''')

for row in all:
    print(row)

conn.close() # Adding conn.close() after the loop

('75267501', '2025-12-21', 'https://archiveofourown.org/works/75267501', 'Scraped', 'the wolf and the fox', 'GoddessOfWriting , TheDauntless (GoddessOfWriting)', '["Mature"]', '["F/M"]', '["A Song of Ice and Fire - George R. R. Martin", "A Song of Ice and Fire & Related Fandoms", "Game of Thrones (TV)"]', '["Jon Snow/Original Female Character(s)", "Jon Snow & Original Female Character(s)", "Jon Snow & Arya Stark & Bran Stark & Rickon Stark & Robb Stark & Sansa Stark", "Ghost | Jon Snow\'s Direwolf & Jon Snow", "Catelyn Tully Stark/Ned Stark", "Jon Snow & Ned Stark"]', '["Original Female Character(s)", "Characters From The North (A Song of Ice and Fire)", "Jon Snow", "Ghost | Jon Snow\'s Direwolf", "Arya Stark", "Nymeria | Arya Stark\'s Direwolf", "Sansa Stark", "Robb Stark", "Theon Greyjoy", "Bran Stark", "Rickon Stark", "Summer | Bran Stark\'s Direwolf", "Shaggydog | Rickon Stark\'s Direwolf", "Ned Stark", "Catelyn Tully Stark", "Maester Luwin of Winterfell (A Song of Ice and Fire)", 

In [ ]:
conn = sqlite3.connect('ao3_works.db')
c = conn.cursor()

# overdue_link_rows = c.execute('''
#       Select link from all_works
#       where published < date('now', '-1 days')
#       and status = 'New'
# ''').fetchall()

overdue_link_rows = c.execute('''
      Select link from all_works
      where status = 'New'
''').fetchall()

overdue_link_list = [row[0] for row in overdue_link_rows]

conn.close()
overdue_link_list

['https://archiveofourown.org/works/84723811',
 'https://archiveofourown.org/works/84715966',
 'https://archiveofourown.org/works/84714186',
 'https://archiveofourown.org/works/84711981',
 'https://archiveofourown.org/works/84711101',
 'https://archiveofourown.org/works/84705176',
 'https://archiveofourown.org/works/84702331',
 'https://archiveofourown.org/works/84702261',
 'https://archiveofourown.org/works/84761471',
 'https://archiveofourown.org/works/84759696',
 'https://archiveofourown.org/works/84758166',
 'https://archiveofourown.org/works/84757686',
 'https://archiveofourown.org/works/84755856',
 'https://archiveofourown.org/works/84751816',
 'https://archiveofourown.org/works/84751086',
 'https://archiveofourown.org/works/84749796',
 'https://archiveofourown.org/works/84741426']

In [ ]:
conn = sqlite3.connect('ao3_works.db')
c = conn.cursor()

all = c.execute('''
      Select *
      from public_bookmarks
      limit 20
''')

for row in all:
    print(row)

conn.close() # Adding conn.close() after the loop

('AmethystQuil', '75267501')
('waometh', '75267501')
('Prpledinosaur', '75267501')
('Sandy_veiw', '75267501')
('Dragon_of_Thunder', '75267501')
('Plopp00', '75267501')
('Hewn820', '75267501')
('WinterAutumn102', '75267501')
('ch_ch_ch_cherrybomb', '75267501')
('username0020', '75267501')
('Rene7968', '75267501')
('procrasinator', '75267501')
('Sister0Pirate0Queen0Mary0Read', '75267501')
('Hush18', '75267501')
('Cribbit', '75267501')
('Reddfoxx297', '75267501')
('Kiro2001', '75267501')
('marytutor', '75267501')
('Murnix', '75267501')
('Ella05', '75267501')


Collaborative filtering using my_bookmarks and public_bookmarks

In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("ao3_works.db")

df_my_bookmarks = pd.read_sql_query("SELECT work_id FROM my_bookmarks", conn)
df_public_bookmarks = pd.read_sql_query("SELECT user_id, work_id FROM public_bookmarks", conn)
conn.close()

In [ ]:
df_my_bookmarks

,work_id
0,75267501
1,56181721
2,81993896
3,76423276
4,55165036
5,50992918
6,73894251
7,62910172


In [ ]:
df_public_bookmarks

,user_id,work_id
0,AmethystQuil,75267501
1,waometh,75267501
2,Prpledinosaur,75267501
3,Sandy_veiw,75267501
4,Dragon_of_Thunder,75267501
...,...,...
2086,yilzmff,84738976
2087,ValyrianScribe,84738976
2088,Theunlickystar,84738976
2089,3efirka,84737881


In [ ]:
my_user_id = "-1"

df_my_bookmarks["user_id"] = my_user_id
df_my_bookmarks["interaction_score"] = 1

df_public_bookmarks["interaction_score"] = 1

In [ ]:
df_my_bookmarks = df_my_bookmarks[["user_id", "work_id", "interaction_score"]]
df_my_bookmarks

,user_id,work_id,interaction_score
0,-1,75267501,1
1,-1,56181721,1
2,-1,81993896,1
3,-1,76423276,1
4,-1,55165036,1
5,-1,50992918,1
6,-1,73894251,1
7,-1,62910172,1


In [ ]:
df_public_bookmarks = df_public_bookmarks[["user_id", "work_id", "interaction_score"]]
df_public_bookmarks

,user_id,work_id,interaction_score
0,AmethystQuil,75267501,1
1,waometh,75267501,1
2,Prpledinosaur,75267501,1
3,Sandy_veiw,75267501,1
4,Dragon_of_Thunder,75267501,1
...,...,...,...
2086,yilzmff,84738976,1
2087,ValyrianScribe,84738976,1
2088,Theunlickystar,84738976,1
2089,3efirka,84737881,1


In [ ]:
df_interactions = pd.concat(
    [df_my_bookmarks, df_public_bookmarks],
    ignore_index=True
)
df_interactions

,user_id,work_id,interaction_score
0,-1,75267501,1
1,-1,56181721,1
2,-1,81993896,1
3,-1,76423276,1
4,-1,55165036,1
...,...,...,...
2094,yilzmff,84738976,1
2095,ValyrianScribe,84738976,1
2096,Theunlickystar,84738976,1
2097,3efirka,84737881,1


In [ ]:
num_unique_work_ids = df_interactions['work_id'].nunique()
print(f"Number of unique work IDs in df_interactions: {num_unique_work_ids}")

Number of unique work IDs in df_interactions: 40


In [ ]:
import pandas as pd
from scipy.sparse import coo_matrix


def create_user_work_matrix(df: pd.DataFrame):
    """
    Creates a sparse user-work interaction matrix for AO3 bookmarks.

    Rows = users/bookmarkers
    Columns = AO3 works
    Values = interaction_score, usually 1 for bookmarked
    """

    df = df.copy()

    # Make sure IDs are strings
    df["user_id"] = df["user_id"].astype(str)
    df["work_id"] = df["work_id"].astype(str)

    # Make sure score is numeric
    df["interaction_score"] = pd.to_numeric(df["interaction_score"])

    # Remove duplicate user-work pairs
    # If duplicates exist, keep the max score
    df = (
        df.groupby(["user_id", "work_id"], as_index=False)
        ["interaction_score"]
        .max()
    )

    user_mapper = {user_id: i for i, user_id in enumerate(df["user_id"].unique())}
    work_mapper = {work_id: i for i, work_id in enumerate(df["work_id"].unique())}
    print(type(user_mapper))
    print(type(work_mapper))

    user_index = df["user_id"].map(user_mapper).values
    work_index = df["work_id"].map(work_mapper).values
    print(type(user_index))
    print(type(work_index))

    X = coo_matrix(
        (
            df["interaction_score"].values,
            (user_index, work_index)
        ),
        shape=(len(user_mapper), len(work_mapper))
    )

    return X.tocsr(), user_mapper, work_mapper

In [ ]:
X, user_mapper, work_mapper = create_user_work_matrix(df_interactions)

print(X.shape)
print(user_mapper["-1"])

<class 'dict'>
<class 'dict'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
(1682, 40)
0


In [ ]:
n_total = X.shape[0]*X.shape[1]
n_ratings = X.nnz
sparsity = n_ratings/n_total
print(f"Matrix percentage filled: {round(sparsity*100,2)}% \nMatrix percentage empty: {100-round(sparsity*100,2)}%")

Matrix percentage filled: 3.12% 
Matrix percentage empty: 96.88%


In [ ]:
# find recoommendations using KNN item based collaborative filtering
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors


def recommend_works_item_based(
    user_id,
    X,
    user_mapper,
    work_mapper,
    k=10,
    metric="cosine",
    min_similarity=0.01
):
    """
    Item-based collaborative filtering for AO3.

    X shape:
        users × works

    item_user_matrix shape:
        works × users

    Scores are weighted by actual cosine similarity.
    """

    user_id = str(user_id)

    if user_id not in user_mapper:
        print(f"User ID {user_id} not found.")
        return pd.DataFrame(columns=["work_id", "score"])

    item_user_matrix = X.T

    user_index = user_mapper[user_id]
    user_interactions = X[user_index]

    already_seen = set(user_interactions.indices)

    n_works = item_user_matrix.shape[0]

    if n_works == 0:
        return pd.DataFrame(columns=["work_id", "score"])

    n_neighbors = min(k + 1, n_works)

    kNN = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric=metric
    )

    kNN.fit(item_user_matrix)

    scores = np.zeros(n_works)
    max_similarity = np.zeros(n_works)
    matched_seen_works = {i: [] for i in range(n_works)}

    for seen_work_index in user_interactions.indices:
        seen_work_vec = item_user_matrix[seen_work_index]

        distances, similar_work_indices = kNN.kneighbors(
            seen_work_vec,
            return_distance=True
        )

        distances = distances.flatten()
        similar_work_indices = similar_work_indices.flatten()

        original_score = user_interactions[0, seen_work_index]

        for similar_work_index, distance in zip(similar_work_indices, distances):
            if similar_work_index == seen_work_index:
                continue

            similarity = 1 - distance

            if similarity < min_similarity:
                continue

            scores[similar_work_index] += original_score * similarity
            max_similarity[similar_work_index] = max(
                max_similarity[similar_work_index],
                similarity
            )
            matched_seen_works[similar_work_index].append(seen_work_index)

    recommended_indices = [
        i for i in np.argsort(-scores)
        if i not in already_seen and scores[i] > 0
    ]

    index_to_work = {
        index: work_id
        for work_id, index in work_mapper.items()
    }

    recommendations = [
        {
            "work_id": index_to_work[i],
            "score": scores[i],
            "max_similarity": max_similarity[i],
            "matched_profile_works": [
                index_to_work[j] for j in matched_seen_works[i]
            ]
        }
        for i in recommended_indices
    ]

    return pd.DataFrame(recommendations)

In [ ]:
recommendations = recommend_works_item_based(
    user_id="-1",
    X=X,
    user_mapper=user_mapper,
    work_mapper=work_mapper,
    k=20,
    min_similarity=0.01
)

In [ ]:
recommendations

,work_id,score,max_similarity,matched_profile_works
0,84738976,0.086367,0.045644,"[75267501, 81993896]"
1,84763896,0.027821,0.027821,[62910172]


In [ ]:
# find recoommendations using cosine_similarity item based collaborative filtering
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


def recommend_works_item_based(
    user_id,
    X,
    user_mapper,
    work_mapper,
    min_similarity=0.01
):


    user_id = str(user_id)

    if user_id not in user_mapper:
        print(f"User ID {user_id} not found.")
        return pd.DataFrame(
            columns=[
                "work_id",
                "collab_score",
                "collab_score_sum",
                "max_similarity",
                "matched_profile_work_count",
                "matched_profile_works"
            ]
        )

    # Convert users × works into works × users
    item_user_matrix = X.T

    user_index = user_mapper[user_id]
    user_interactions = X[user_index]

    # Works already bookmarked/read by the user
    bookmarked_work_indices = user_interactions.indices
    bookmarked_work_set = set(bookmarked_work_indices)

    if len(bookmarked_work_indices) == 0:
        return pd.DataFrame(
            columns=[
                "work_id",
                "collab_score",
                "collab_score_sum",
                "max_similarity",
                "matched_profile_work_count",
                "matched_profile_works"
            ]
        )

    # All unread candidate works
    candidate_work_indices = np.array([
        i for i in range(item_user_matrix.shape[0])
        if i not in bookmarked_work_set
    ])

    if len(candidate_work_indices) == 0:
        return pd.DataFrame(
            columns=[
                "work_id",
                "collab_score",
                "collab_score_sum",
                "max_similarity",
                "matched_profile_work_count",
                "matched_profile_works"
            ]
        )

    bookmarked_matrix = item_user_matrix[bookmarked_work_indices]
    candidate_matrix = item_user_matrix[candidate_work_indices]

    # Shape:
    # candidates × bookmarked works
    similarity_matrix = cosine_similarity(
        candidate_matrix,
        bookmarked_matrix
    )

    # Ignore tiny/noisy similarity values
    similarity_matrix[similarity_matrix < min_similarity] = 0

    # Aggregate similarity signals
    collab_score_sum = similarity_matrix.sum(axis=1)

    # Mean score stays on a more comparable 0–1-ish scale
    collab_score_mean = similarity_matrix.mean(axis=1)

    max_similarity = similarity_matrix.max(axis=1)

    matched_profile_work_count = (
        similarity_matrix > 0
    ).sum(axis=1)

    index_to_work = {
        index: work_id
        for work_id, index in work_mapper.items()
    }

    recommendations = []

    for row_idx, candidate_work_index in enumerate(candidate_work_indices):
        matched_indices = np.where(similarity_matrix[row_idx] > 0)[0]

        matched_profile_works = [
            index_to_work[bookmarked_work_indices[j]]
            for j in matched_indices
        ]

        recommendations.append({
            "work_id": index_to_work[candidate_work_index],
            "collab_score": collab_score_mean[row_idx],
            "collab_score_sum": collab_score_sum[row_idx],
            "max_similarity": max_similarity[row_idx],
            "matched_profile_work_count": matched_profile_work_count[row_idx],
            "matched_profile_works": matched_profile_works
        })

    recommendations_df = pd.DataFrame(recommendations)

    recommendations_df = recommendations_df.sort_values(
        by=["collab_score_sum", "max_similarity"],
        ascending=False
    ).reset_index(drop=True)

    return recommendations_df

In [ ]:
collab_recommendations = recommend_works_item_based(
    user_id="-1",
    X=X,
    user_mapper=user_mapper,
    work_mapper=work_mapper,
    min_similarity=0.01
)

collab_recommendations.head(20)

,work_id,collab_score,collab_score_sum,max_similarity,matched_profile_work_count,matched_profile_works
0,84738976,0.010796,0.086367,0.045644,2,"[75267501, 81993896]"
1,84763896,0.003478,0.027821,0.027821,1,[62910172]
2,84737881,0.000000,0.000000,0.000000,0,[]
3,84752771,0.000000,0.000000,0.000000,0,[]
4,84711126,0.000000,0.000000,0.000000,0,[]
5,84721256,0.000000,0.000000,0.000000,0,[]
6,84714196,0.000000,0.000000,0.000000,0,[]
7,84744936,0.000000,0.000000,0.000000,0,[]
8,84745361,0.000000,0.000000,0.000000,0,[]
9,84743746,0.000000,0.000000,0.000000,0,[]


In [ ]:
collab_recommendations['collab_score_sum_normalized'] = (
    collab_recommendations["collab_score_sum"]
    / collab_recommendations["collab_score_sum"].max()
)
collab_recommendations

,work_id,collab_score,collab_score_sum,max_similarity,matched_profile_work_count,matched_profile_works,collab_score_sum_normalized
0,84738976,0.010796,0.086367,0.045644,2,"[75267501, 81993896]",1.000000
1,84763896,0.003478,0.027821,0.027821,1,[62910172],0.322124
2,84737881,0.000000,0.000000,0.000000,0,[],0.000000
3,84752771,0.000000,0.000000,0.000000,0,[],0.000000
4,84711126,0.000000,0.000000,0.000000,0,[],0.000000
5,84721256,0.000000,0.000000,0.000000,0,[],0.000000
6,84714196,0.000000,0.000000,0.000000,0,[],0.000000
7,84744936,0.000000,0.000000,0.000000,0,[],0.000000
8,84745361,0.000000,0.000000,0.000000,0,[],0.000000
9,84743746,0.000000,0.000000,0.000000,0,[],0.000000


In [ ]:
import sqlite3

conn = sqlite3.connect('ao3_works.db')
c = conn.cursor()

# Get the list of recommended work IDs
recommendation_work_ids = collab_recommendations['work_id'].tolist()

# Create a string of placeholders for the IN clause (e.g., '?,?,?')
placeholders = ','.join('?' * len(recommendation_work_ids))

# Construct the query with dynamic placeholders
query = f"SELECT * FROM all_works WHERE id IN ({placeholders})"

# Execute the query, passing the list of work IDs as parameters
all_works_from_recommendations = c.execute(query, recommendation_work_ids).fetchall()

for row in all_works_from_recommendations:
    print(row)

conn.close()

('84696726', '2026-05-12T00:40:29Z', 'https://archiveofourown.org/works/84696726', 'Scraped', 'HOTD Hogwarts Modern AU', 'Lying_on_floors_12', '["General Audiences"]', '["Gen"]', '["House of the Dragon (TV)"]', '["Aemond \\"One-Eye\\" Targaryen & Helaena Targaryen", "Aegon II Targaryen & Aemond \\"One-Eye\\" Targaryen", "Alicent Hightower & Aemond \\"One-Eye\\" Targaryen", "Otto Hightower & Aemond \\"One-Eye\\" Targaryen"]', '["Aemond \\"One-Eye\\" Targaryen", "Helaena Targaryen", "Aegon II Targaryen", "Alicent Hightower", "Otto Hightower", "Aemond \\"One-Eye\\" Targaryen ", " Helaena Targaryen", "Aegon II Targaryen ", " Aemond \\"One-Eye\\" Targaryen", "Alicent Hightower ", "Otto Hightower "]', '["Alternate Universe - Hogwarts", "Aegon II Targaryen is Not a Rapist", "Family Fluff", "House Targaryen (A Song of Ice and Fire)", "Sibling Bonding", "Quidditch", "Alternate Universe - Modern Setting", "Alternate Universe - Modern with Magic"]', 'A collection of Hogwarts based stories of the 

Content based filtering using tags of my bookmarked works and all works

In [ ]:
import sqlite3

conn = sqlite3.connect('ao3_works.db')
c = conn.cursor()

bookmarked_works_df = pd.read_sql_query('''
          Select id, rating, categories, fandoms, relationships, characters, freeform_tags
          from all_works
          where id in (
                            Select work_id
                            from my_bookmarks
                      )
          and status = "Scraped"
          ''',
          conn)


not_bookmarked_works_df = pd.read_sql_query('''
          Select id, rating, categories, fandoms, relationships, characters, freeform_tags
          from all_works
          where id not in (
                            Select work_id
                            from my_bookmarks
                      )
          and status = "Scraped"
          ''',
          conn)

conn.close()

In [ ]:
bookmarked_works_df

,id,rating,categories,fandoms,relationships,characters,freeform_tags
0,50992918,"[""Mature""]","[""F/F"", ""F/M"", ""Gen"", ""Multi""]","[""A Song of Ice and Fire - George R. R. Martin...","[""Jon Snow/Sansa Stark"", ""Jon Snow/Daenerys Ta...","[""Daenerys Targaryen"", ""Jon Snow"", ""Sansa Star...","[""R Plus L Equals J"", ""Prophetic dreams someti..."
1,55165036,"[""Mature""]",[],"[""A Song of Ice and Fire & Related Fandoms"", ""...","[""Ned Stark/Shaena Targaryen"", ""Catelyn Tully ...","[""Ned Stark"", ""Rickard Stark"", ""Brandon Stark""...","[""Ned Stark Needs a Hug"", ""Alternate Universe ..."
2,56181721,"[""Mature""]","[""Gen"", ""F/F"", ""F/M""]","[""A Song of Ice and Fire - George R. R. Martin...","[""Elia Martell/Rhaegar Targaryen"", ""Ashara Day...","[""Rhaegar Targaryen"", ""Aerys II Targaryen"", ""T...","[""Alternate Universe - Canon Divergence"", ""No ..."
3,62910172,"[""Mature""]","[""F/M""]","[""A Song of Ice and Fire - George R. R. Martin""]","[""Jon Snow & Rhaegar Targaryen"", ""Elia Martell...","[""Jon Snow"", ""Daenerys Targaryen"", ""Rhaenys Ta...","[""King Rhaegar Targaryen"", ""Robert's Rebellion..."
4,73894251,"[""Mature""]","[""F/F"", ""F/M""]","[""A Song of Ice and Fire - George R. R. Martin...","[""Jon Snow/Sansa Stark"", ""Jon Snow/Daenerys Ta...","[""Jon Snow"", ""Daenerys Targaryen"", ""Sansa Star...","[""R Plus L Equals J | Lyanna Stark and Rhaegar..."
5,75267501,"[""Mature""]","[""F/M""]","[""A Song of Ice and Fire - George R. R. Martin...","[""Jon Snow/Original Female Character(s)"", ""Jon...","[""Original Female Character(s)"", ""Characters F...","[""Original Noble House (A Song of Ice and Fire..."
6,76423276,"[""Mature""]","[""F/M""]","[""A Song of Ice and Fire - George R. R. Martin...","[""Cersei Lannister/Jon Snow"", ""Ashara Dayne/Jo...","[""Jon Snow"", ""Cersei Lannister"", ""Tyrion Lanni...","[""Alternate Universe - Canon Divergence"", ""Jon..."
7,81993896,"[""Mature""]","[""F/M"", ""Gen""]","[""Game of Thrones (TV)"", ""A Song of Ice and Fi...","[""Jon Snow/Margaery Tyrell"", ""Ysilla Royce/Rob...","[""Jon Snow"", ""Robb Stark"", ""Ned Stark"", ""Catel...","[""Action/Adventure"", ""Action & Romance"", ""Cano..."


In [ ]:
not_bookmarked_works_df

,id,rating,categories,fandoms,relationships,characters,freeform_tags
0,84723496,"[""Mature""]","[""F/M""]","[""A Song of Ice and Fire & Related Fandoms"", ""...","[""Original Blackfyre Character/Ashara Dayne""]","[""Original Blackfyre Character(s)"", ""Original ...","[""Alternate Universe - Canon Divergence"", ""Hou..."
1,84723001,"[""General Audiences""]","[""F/M""]","[""Game of Thrones (TV)""]","[""Jaime Lannister/Brienne of Tarth""]","[""Jaime Lannister"", ""Brienne of Tarth""]","[""Banter"", ""Mild Hurt/Comfort"", ""Short"", ""Perm..."
2,84722136,"[""Mature""]","[""F/F"", ""F/M""]","[""A Song of Ice and Fire - George R. R. Martin...",[],[],[]
3,84721256,"[""Mature""]","[""F/F"", ""F/M"", ""M/M""]","[""House of the Dragon (TV)""]","[""Aemond \""One-Eye\"" Targaryen/Original Female...","[""Aemond \""One-Eye\"" Targaryen"", ""Helaena Targ...","[""Alicent Hightower & Rhaenyra Targaryen Frien..."
4,84720706,"[""Mature""]","[""M/M""]","[""A Knight of the Seven Kingdoms (TV)"", ""A Son...","[""Dunk | Duncan the Tall/Aerion Targaryen (Son...","[""Dunk | Duncan the Tall (A Song of Ice and Fi...","[""Murder Mystery"", ""Police Uniforms"", ""Alterna..."
5,84719016,"[""Mature""]","[""F/M""]","[""A Knight of the Seven Kingdoms (TV)""]","[""Aerion Targaryen (Son of Maekar I)/You""]","[""Aerion Targaryen (Son of Maekar I)"", ""Maegor...","[""Female Reader-Insert"", ""No Use of Y/N for Re..."
6,84715021,"[""Not Rated""]","[""F/M""]","[""A Knight of the Seven Kingdoms (TV)"", ""A Son...","[""Baelor \""Breakspear\"" Targaryen/Original Fem...","[""Baelor \""Breakspear\"" Targaryen"", ""Original ...","[""Female Knight"", ""False knight"", ""Mystery kni..."
7,84714196,"[""Teen And Up Audiences""]","[""Gen""]","[""A Song of Ice and Fire - George R. R. Martin...","[""Dunk | Duncan the Tall & Aegon V \""Egg\"" Tar...","[""Dunk | Duncan the Tall (A Song of Ice and Fi...","[""Retelling"", ""POV Aegon V \""Egg\"" Targaryen""]"
8,84711126,"[""General Audiences""]","[""F/M"", ""Gen""]","[""House of the Dragon (TV)"", ""Marvel""]","[""Alicent Hightower & Aegon II Targaryen"", ""Al...","[""Alicent Hightower"", ""Wanda Maximoff"", ""Gwayn...","[""House Hightower (A Song of Ice and Fire)"", ""..."
9,84709216,"[""Mature""]","[""F/M""]","[""A Knight of the Seven Kingdoms (TV)"", ""A Son...","[""Aerion Targaryen (Son of Maekar I)/You""]","[""Aerion Targaryen (Son of Maekar I)"", ""Reader...","[""Established Relationship"", ""Fluff and Angst""..."


In [ ]:
import json
import pandas as pd

# Weight for each tag type in the similarity calculation
tag_cols = {
    "rating": 0.5,
    "categories": 0.6,
    "fandoms": 0.4,
    "relationships": 1,
    "characters": 0.9,
    "freeform_tags": 0.9
}

def parse_json_list(value):
    if pd.isna(value) or value == "":
        return []

    values = json.loads(value)

    # clean whitespace + remove duplicates within that same cell
    cleaned = [str(v).strip() for v in values if str(v).strip()]
    return list(dict.fromkeys(cleaned))

In [ ]:
not_bookmarked_works_df

copy_of_not_bookmarked_works_df = not_bookmarked_works_df.copy()

for col in tag_cols.keys():
    copy_of_not_bookmarked_works_df[col] = copy_of_not_bookmarked_works_df[col].apply(parse_json_list)
copy_of_not_bookmarked_works_df

,id,rating,categories,fandoms,relationships,characters,freeform_tags
0,84723496,[Mature],[F/M],"[A Song of Ice and Fire & Related Fandoms, Gam...",[Original Blackfyre Character/Ashara Dayne],"[Original Blackfyre Character(s), Original Mal...","[Alternate Universe - Canon Divergence, House ..."
1,84723001,[General Audiences],[F/M],[Game of Thrones (TV)],[Jaime Lannister/Brienne of Tarth],"[Jaime Lannister, Brienne of Tarth]","[Banter, Mild Hurt/Comfort, Short, Permanent I..."
2,84722136,[Mature],"[F/F, F/M]","[A Song of Ice and Fire - George R. R. Martin,...",[],[],[]
3,84721256,[Mature],"[F/F, F/M, M/M]",[House of the Dragon (TV)],"[Aemond ""One-Eye"" Targaryen/Original Female Ch...","[Aemond ""One-Eye"" Targaryen, Helaena Targaryen...",[Alicent Hightower & Rhaenyra Targaryen Friend...
4,84720706,[Mature],[M/M],"[A Knight of the Seven Kingdoms (TV), A Song o...",[Dunk | Duncan the Tall/Aerion Targaryen (Son ...,[Dunk | Duncan the Tall (A Song of Ice and Fir...,"[Murder Mystery, Police Uniforms, Alternate Un..."
5,84719016,[Mature],[F/M],[A Knight of the Seven Kingdoms (TV)],[Aerion Targaryen (Son of Maekar I)/You],"[Aerion Targaryen (Son of Maekar I), Maegor Ta...","[Female Reader-Insert, No Use of Y/N for Reade..."
6,84715021,[Not Rated],[F/M],"[A Knight of the Seven Kingdoms (TV), A Song o...","[Baelor ""Breakspear"" Targaryen/Original Female...","[Baelor ""Breakspear"" Targaryen, Original Femal...","[Female Knight, False knight, Mystery knight, ..."
7,84714196,[Teen And Up Audiences],[Gen],"[A Song of Ice and Fire - George R. R. Martin,...","[Dunk | Duncan the Tall & Aegon V ""Egg"" Targar...",[Dunk | Duncan the Tall (A Song of Ice and Fir...,"[Retelling, POV Aegon V ""Egg"" Targaryen]"
8,84711126,[General Audiences],"[F/M, Gen]","[House of the Dragon (TV), Marvel]","[Alicent Hightower & Aegon II Targaryen, Alice...","[Alicent Hightower, Wanda Maximoff, Gwayne Hig...","[House Hightower (A Song of Ice and Fire), Mag..."
9,84709216,[Mature],[F/M],"[A Knight of the Seven Kingdoms (TV), A Song o...",[Aerion Targaryen (Son of Maekar I)/You],"[Aerion Targaryen (Son of Maekar I), Reader, You]","[Established Relationship, Fluff and Angst, Ma..."


In [ ]:
copy_of_bookmarked_works_df = bookmarked_works_df.copy()

for col in tag_cols.keys():
    copy_of_bookmarked_works_df[col] = copy_of_bookmarked_works_df[col].apply(parse_json_list)
copy_of_bookmarked_works_df

,id,rating,categories,fandoms,relationships,characters,freeform_tags
0,50992918,[Mature],"[F/F, F/M, Gen, Multi]","[A Song of Ice and Fire - George R. R. Martin,...","[Jon Snow/Sansa Stark, Jon Snow/Daenerys Targa...","[Daenerys Targaryen, Jon Snow, Sansa Stark, Ne...","[R Plus L Equals J, Prophetic dreams sometimes..."
1,55165036,[Mature],[],"[A Song of Ice and Fire & Related Fandoms, A S...","[Ned Stark/Shaena Targaryen, Catelyn Tully Sta...","[Ned Stark, Rickard Stark, Brandon Stark, Shae...","[Ned Stark Needs a Hug, Alternate Universe - C..."
2,56181721,[Mature],"[Gen, F/F, F/M]","[A Song of Ice and Fire - George R. R. Martin,...","[Elia Martell/Rhaegar Targaryen, Ashara Dayne/...","[Rhaegar Targaryen, Aerys II Targaryen, Tywin ...","[Alternate Universe - Canon Divergence, No Rob..."
3,62910172,[Mature],[F/M],[A Song of Ice and Fire - George R. R. Martin],"[Jon Snow & Rhaegar Targaryen, Elia Martell/Rh...","[Jon Snow, Daenerys Targaryen, Rhaenys Targary...","[King Rhaegar Targaryen, Robert's Rebellion Fa..."
4,73894251,[Mature],"[F/F, F/M]","[A Song of Ice and Fire - George R. R. Martin,...","[Jon Snow/Sansa Stark, Jon Snow/Daenerys Targa...","[Jon Snow, Daenerys Targaryen, Sansa Stark, Ne...",[R Plus L Equals J | Lyanna Stark and Rhaegar ...
5,75267501,[Mature],[F/M],"[A Song of Ice and Fire - George R. R. Martin,...","[Jon Snow/Original Female Character(s), Jon Sn...","[Original Female Character(s), Characters From...",[Original Noble House (A Song of Ice and Fire)...
6,76423276,[Mature],[F/M],"[A Song of Ice and Fire - George R. R. Martin,...","[Cersei Lannister/Jon Snow, Ashara Dayne/Jon S...","[Jon Snow, Cersei Lannister, Tyrion Lannister,...","[Alternate Universe - Canon Divergence, Jon Sn..."
7,81993896,[Mature],"[F/M, Gen]","[Game of Thrones (TV), A Song of Ice and Fire ...","[Jon Snow/Margaery Tyrell, Ysilla Royce/Robb S...","[Jon Snow, Robb Stark, Ned Stark, Catelyn Tull...","[Action/Adventure, Action & Romance, Canon - A..."


In [ ]:
copy_of_bookmarked_works_df["is_bookmarked"] = 1
copy_of_not_bookmarked_works_df["is_bookmarked"] = 0

In [ ]:
all_works_df = pd.concat(
    [copy_of_bookmarked_works_df, copy_of_not_bookmarked_works_df],
    ignore_index=True
)
all_works_df

,id,rating,categories,fandoms,relationships,characters,freeform_tags,is_bookmarked
0,50992918,[Mature],"[F/F, F/M, Gen, Multi]","[A Song of Ice and Fire - George R. R. Martin,...","[Jon Snow/Sansa Stark, Jon Snow/Daenerys Targa...","[Daenerys Targaryen, Jon Snow, Sansa Stark, Ne...","[R Plus L Equals J, Prophetic dreams sometimes...",1
1,55165036,[Mature],[],"[A Song of Ice and Fire & Related Fandoms, A S...","[Ned Stark/Shaena Targaryen, Catelyn Tully Sta...","[Ned Stark, Rickard Stark, Brandon Stark, Shae...","[Ned Stark Needs a Hug, Alternate Universe - C...",1
2,56181721,[Mature],"[Gen, F/F, F/M]","[A Song of Ice and Fire - George R. R. Martin,...","[Elia Martell/Rhaegar Targaryen, Ashara Dayne/...","[Rhaegar Targaryen, Aerys II Targaryen, Tywin ...","[Alternate Universe - Canon Divergence, No Rob...",1
3,62910172,[Mature],[F/M],[A Song of Ice and Fire - George R. R. Martin],"[Jon Snow & Rhaegar Targaryen, Elia Martell/Rh...","[Jon Snow, Daenerys Targaryen, Rhaenys Targary...","[King Rhaegar Targaryen, Robert's Rebellion Fa...",1
4,73894251,[Mature],"[F/F, F/M]","[A Song of Ice and Fire - George R. R. Martin,...","[Jon Snow/Sansa Stark, Jon Snow/Daenerys Targa...","[Jon Snow, Daenerys Targaryen, Sansa Stark, Ne...",[R Plus L Equals J | Lyanna Stark and Rhaegar ...,1
5,75267501,[Mature],[F/M],"[A Song of Ice and Fire - George R. R. Martin,...","[Jon Snow/Original Female Character(s), Jon Sn...","[Original Female Character(s), Characters From...",[Original Noble House (A Song of Ice and Fire)...,1
6,76423276,[Mature],[F/M],"[A Song of Ice and Fire - George R. R. Martin,...","[Cersei Lannister/Jon Snow, Ashara Dayne/Jon S...","[Jon Snow, Cersei Lannister, Tyrion Lannister,...","[Alternate Universe - Canon Divergence, Jon Sn...",1
7,81993896,[Mature],"[F/M, Gen]","[Game of Thrones (TV), A Song of Ice and Fire ...","[Jon Snow/Margaery Tyrell, Ysilla Royce/Robb S...","[Jon Snow, Robb Stark, Ned Stark, Catelyn Tull...","[Action/Adventure, Action & Romance, Canon - A...",1
8,84723496,[Mature],[F/M],"[A Song of Ice and Fire & Related Fandoms, Gam...",[Original Blackfyre Character/Ashara Dayne],"[Original Blackfyre Character(s), Original Mal...","[Alternate Universe - Canon Divergence, House ...",0
9,84723001,[General Audiences],[F/M],[Game of Thrones (TV)],[Jaime Lannister/Brienne of Tarth],"[Jaime Lannister, Brienne of Tarth]","[Banter, Mild Hurt/Comfort, Short, Permanent I...",0


In [ ]:
work_mapper = {work_id: i for i, work_id in enumerate(all_works_df["id"].unique())}
work_index = all_works_df["id"].map(work_mapper).values

In [ ]:
rows = []
cols = []
data = []

tag_mapper = {}
next_tag_idx = 0

for _, row in all_works_df.iterrows():
    work_idx = work_mapper[row["id"]]

    for col_name, weight in tag_cols.items():
        for tag in row[col_name]:
            feature_name = f"{col_name}::{tag}"

            if feature_name not in tag_mapper:
                tag_mapper[feature_name] = next_tag_idx
                next_tag_idx += 1

            tag_idx = tag_mapper[feature_name]

            rows.append(work_idx)
            cols.append(tag_idx)
            data.append(weight)

In [ ]:
tag_mapper

{'rating::Mature': 0,
 'categories::F/F': 1,
 'categories::F/M': 2,
 'categories::Gen': 3,
 'categories::Multi': 4,
 'fandoms::A Song of Ice and Fire - George R. R. Martin': 5,
 'fandoms::Game of Thrones (TV)': 6,
 'fandoms::A Song of Ice and Fire & Related Fandoms': 7,
 'relationships::Jon Snow/Sansa Stark': 8,
 'relationships::Jon Snow/Daenerys Targaryen': 9,
 'relationships::Sansa Stark/Daenerys Targaryen': 10,
 'characters::Daenerys Targaryen': 11,
 'characters::Jon Snow': 12,
 'characters::Sansa Stark': 13,
 'characters::Ned Stark': 14,
 'characters::Robb Stark': 15,
 'characters::Catelyn Tully Stark': 16,
 'characters::Bran Stark': 17,
 'characters::Brynden "Bloodraven" Rivers': 18,
 'characters::Young Griff (ASoIaF)': 19,
 'freeform_tags::R Plus L Equals J': 20,
 'freeform_tags::Prophetic dreams sometimes creepy sometimes not': 21,
 'freeform_tags::what is a threesome?': 22,
 'freeform_tags::Sometimes I just steal whole sentences from the book lmao': 23,
 'freeform_tags::Jonsa s

In [ ]:
from scipy.sparse import coo_matrix
work_tag_matrix = coo_matrix(
    (data, (rows, cols)),
    shape=(len(work_mapper), len(tag_mapper))
)
work_tag_matrix = work_tag_matrix.tocsr()

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

bookmark_mask = all_works_df["is_bookmarked"].to_numpy() == 1
candidate_mask = all_works_df["is_bookmarked"].to_numpy() == 0

bookmarked_matrix = work_tag_matrix[bookmark_mask]
candidate_matrix = work_tag_matrix[candidate_mask]

In [ ]:
user_profile = np.asarray(bookmarked_matrix.mean(axis=0))

In [ ]:
similarities = cosine_similarity(candidate_matrix, user_profile).ravel()

In [ ]:
recommendations = all_works_df.loc[candidate_mask, ["id"]].copy()
recommendations["similarity_score"] = similarities

recommendations = recommendations.sort_values(
    "similarity_score",
    ascending=False
)

In [ ]:
recommendations

,id,similarity_score
39,84737721,0.327698
8,84723496,0.327024
29,84761796,0.280035
10,84722136,0.273718
21,84701301,0.256271
18,84708881,0.186059
31,84748536,0.162172
30,84752771,0.159890
36,84742296,0.147097
26,84763896,0.146643


In [ ]:
merged_recommendations = pd.merge(recommendations, not_bookmarked_works_df, on='id', how='left')
merged_recommendations.head()

,id,similarity_score,rating,categories,fandoms,relationships,characters,freeform_tags
0,84737721,0.327698,"[""Mature""]","[""F/M"", ""Gen"", ""Other""]","[""A Song of Ice and Fire - George R. R. Martin...","[""Rhaegar Targaryen/Original Female Character""]","[""Rhaegar Targaryen"", ""Ned Stark"", ""Sansa Star...","[""Hurt/Comfort"", ""Angst"", ""Sex"", ""Whimsical"", ..."
1,84723496,0.327024,"[""Mature""]","[""F/M""]","[""A Song of Ice and Fire & Related Fandoms"", ""...","[""Original Blackfyre Character/Ashara Dayne""]","[""Original Blackfyre Character(s)"", ""Original ...","[""Alternate Universe - Canon Divergence"", ""Hou..."
2,84761796,0.280035,"[""Not Rated""]",[],"[""Game of Thrones (TV)"", ""A Song of Ice and Fi...",[],"[""Sansa Stark"", ""Arya Stark"", ""Robb Stark"", ""J...","[""Magic"", ""shortstory"", ""Oneshot"", ""magicmirro..."
3,84722136,0.273718,"[""Mature""]","[""F/F"", ""F/M""]","[""A Song of Ice and Fire - George R. R. Martin...",[],[],[]
4,84701301,0.256271,"[""Mature""]","[""F/M""]","[""Game of Thrones (TV)""]",[],"[""Jon Snow"", ""Ghost | Jon Snow's Direwolf"", ""N...","[""Westeros (A Song of Ice and Fire)"", ""Essos (..."


In [ ]:
merged_recommendations

,id,similarity_score,rating,categories,fandoms,relationships,characters,freeform_tags
0,84737721,0.327698,"[""Mature""]","[""F/M"", ""Gen"", ""Other""]","[""A Song of Ice and Fire - George R. R. Martin...","[""Rhaegar Targaryen/Original Female Character""]","[""Rhaegar Targaryen"", ""Ned Stark"", ""Sansa Star...","[""Hurt/Comfort"", ""Angst"", ""Sex"", ""Whimsical"", ..."
1,84723496,0.327024,"[""Mature""]","[""F/M""]","[""A Song of Ice and Fire & Related Fandoms"", ""...","[""Original Blackfyre Character/Ashara Dayne""]","[""Original Blackfyre Character(s)"", ""Original ...","[""Alternate Universe - Canon Divergence"", ""Hou..."
2,84761796,0.280035,"[""Not Rated""]",[],"[""Game of Thrones (TV)"", ""A Song of Ice and Fi...",[],"[""Sansa Stark"", ""Arya Stark"", ""Robb Stark"", ""J...","[""Magic"", ""shortstory"", ""Oneshot"", ""magicmirro..."
3,84722136,0.273718,"[""Mature""]","[""F/F"", ""F/M""]","[""A Song of Ice and Fire - George R. R. Martin...",[],[],[]
4,84701301,0.256271,"[""Mature""]","[""F/M""]","[""Game of Thrones (TV)""]",[],"[""Jon Snow"", ""Ghost | Jon Snow's Direwolf"", ""N...","[""Westeros (A Song of Ice and Fire)"", ""Essos (..."
5,84708881,0.186059,"[""Mature""]","[""F/M""]","[""Game of Thrones (TV)"", ""A Song of Ice and Fi...","[""Robb Stark/You""]","[""Robb Stark"", ""Catelyn Tully Stark"", ""You""]","[""I miss Robb"", ""Married Life"", ""Not even lowk..."
6,84748536,0.162172,"[""Teen And Up Audiences""]","[""F/M""]","[""Game of Thrones (TV)""]","[""Jorah Mormont/Daenerys Targaryen""]","[""Jorah Mormont"", ""Daenerys Targaryen""]",[]
7,84752771,0.159890,"[""Teen And Up Audiences""]","[""M/M""]","[""A Song of Ice and Fire - George R. R. Martin...","[""Jon Snow/Robb Stark""]","[""Robb Stark"", ""Jon Snow"", ""Grey Wind | Robb S...","[""Drunk Jon Snow"", ""Love Confessions"", ""Drunke..."
8,84742296,0.147097,"[""Teen And Up Audiences""]","[""Other""]","[""Game of Thrones (TV)""]","[""Sansa stark/vengence"", ""Sansa Stark & Lady S...","[""Sansa Stark"", ""Lady Stoneheart"", ""Joffrey Ba...","[""Gun Violence"", ""Revenge"", ""based off of a so..."
9,84763896,0.146643,"[""Mature""]","[""F/M"", ""Gen"", ""M/M""]","[""A Song of Ice and Fire & Related Fandoms"", ""...","[""Lyonel Baratheon/Dunk | Duncan the Tall"", ""O...","[""Aerys II Targaryen"", ""Jaime Lannister"", ""Dun...","[""Parallels"", ""Character Study"", ""Parallels be..."


Final recommendations from tag and collab filters

In [ ]:
final_recommendations = pd.merge(
    merged_recommendations,
    collab_recommendations,
    left_on='id',
    right_on='work_id',
    how='left'
)


# final_recommendations = final_recommendations.sort_values(
#     by=["similarity_score", "collab_score_sum_normalized"],
#     ascending=False
# ).reset_index(drop=True)

final_recommendations.head()

,id,similarity_score,rating,categories,fandoms,relationships,characters,freeform_tags,work_id,collab_score,collab_score_sum,max_similarity,matched_profile_work_count,matched_profile_works,collab_score_sum_normalized
0,84737721,0.327698,"[""Mature""]","[""F/M"", ""Gen"", ""Other""]","[""A Song of Ice and Fire - George R. R. Martin...","[""Rhaegar Targaryen/Original Female Character""]","[""Rhaegar Targaryen"", ""Ned Stark"", ""Sansa Star...","[""Hurt/Comfort"", ""Angst"", ""Sex"", ""Whimsical"", ...",84737721,0.0,0.0,0.0,0,[],0.0
1,84723496,0.327024,"[""Mature""]","[""F/M""]","[""A Song of Ice and Fire & Related Fandoms"", ""...","[""Original Blackfyre Character/Ashara Dayne""]","[""Original Blackfyre Character(s)"", ""Original ...","[""Alternate Universe - Canon Divergence"", ""Hou...",84723496,0.0,0.0,0.0,0,[],0.0
2,84761796,0.280035,"[""Not Rated""]",[],"[""Game of Thrones (TV)"", ""A Song of Ice and Fi...",[],"[""Sansa Stark"", ""Arya Stark"", ""Robb Stark"", ""J...","[""Magic"", ""shortstory"", ""Oneshot"", ""magicmirro...",84761796,0.0,0.0,0.0,0,[],0.0
3,84722136,0.273718,"[""Mature""]","[""F/F"", ""F/M""]","[""A Song of Ice and Fire - George R. R. Martin...",[],[],[],84722136,0.0,0.0,0.0,0,[],0.0
4,84701301,0.256271,"[""Mature""]","[""F/M""]","[""Game of Thrones (TV)""]",[],"[""Jon Snow"", ""Ghost | Jon Snow's Direwolf"", ""N...","[""Westeros (A Song of Ice and Fire)"", ""Essos (...",84701301,0.0,0.0,0.0,0,[],0.0


In [ ]:
final_recommendations['final_score'] = final_recommendations['similarity_score'] * 0.6 + final_recommendations['collab_score_sum_normalized'] * 0.4
final_recommendations.head()

,id,similarity_score,rating,categories,fandoms,relationships,characters,freeform_tags,work_id,collab_score,collab_score_sum,max_similarity,matched_profile_work_count,matched_profile_works,collab_score_sum_normalized,final_score
0,84737721,0.327698,"[""Mature""]","[""F/M"", ""Gen"", ""Other""]","[""A Song of Ice and Fire - George R. R. Martin...","[""Rhaegar Targaryen/Original Female Character""]","[""Rhaegar Targaryen"", ""Ned Stark"", ""Sansa Star...","[""Hurt/Comfort"", ""Angst"", ""Sex"", ""Whimsical"", ...",84737721,0.0,0.0,0.0,0,[],0.0,0.196619
1,84723496,0.327024,"[""Mature""]","[""F/M""]","[""A Song of Ice and Fire & Related Fandoms"", ""...","[""Original Blackfyre Character/Ashara Dayne""]","[""Original Blackfyre Character(s)"", ""Original ...","[""Alternate Universe - Canon Divergence"", ""Hou...",84723496,0.0,0.0,0.0,0,[],0.0,0.196215
2,84761796,0.280035,"[""Not Rated""]",[],"[""Game of Thrones (TV)"", ""A Song of Ice and Fi...",[],"[""Sansa Stark"", ""Arya Stark"", ""Robb Stark"", ""J...","[""Magic"", ""shortstory"", ""Oneshot"", ""magicmirro...",84761796,0.0,0.0,0.0,0,[],0.0,0.168021
3,84722136,0.273718,"[""Mature""]","[""F/F"", ""F/M""]","[""A Song of Ice and Fire - George R. R. Martin...",[],[],[],84722136,0.0,0.0,0.0,0,[],0.0,0.164231
4,84701301,0.256271,"[""Mature""]","[""F/M""]","[""Game of Thrones (TV)""]",[],"[""Jon Snow"", ""Ghost | Jon Snow's Direwolf"", ""N...","[""Westeros (A Song of Ice and Fire)"", ""Essos (...",84701301,0.0,0.0,0.0,0,[],0.0,0.153762


In [ ]:
final_recommendations = final_recommendations.sort_values(
    by=["final_score"],
    ascending=False
).reset_index(drop=True)
final_recommendations

,id,similarity_score,rating,categories,fandoms,relationships,characters,freeform_tags,work_id,collab_score,collab_score_sum,max_similarity,matched_profile_work_count,matched_profile_works,collab_score_sum_normalized,final_score
0,84738976,0.043189,"[""Teen And Up Audiences""]","[""F/M""]","[""A Knight of the Seven Kingdoms (TV)"", ""A Son...","[""Maekar I Targaryen/You""]","[""Maekar I Targaryen"", ""Aegon V \""Egg\"" Targar...","[""Fluff"", ""Angst"", ""Grief/Mourning"", ""nerd rea...",84738976,0.010796,0.086367,0.045644,2,"[75267501, 81993896]",1.000000,0.425913
1,84763896,0.146643,"[""Mature""]","[""F/M"", ""Gen"", ""M/M""]","[""A Song of Ice and Fire & Related Fandoms"", ""...","[""Lyonel Baratheon/Dunk | Duncan the Tall"", ""O...","[""Aerys II Targaryen"", ""Jaime Lannister"", ""Dun...","[""Parallels"", ""Character Study"", ""Parallels be...",84763896,0.003478,0.027821,0.027821,1,[62910172],0.322124,0.216835
2,84737721,0.327698,"[""Mature""]","[""F/M"", ""Gen"", ""Other""]","[""A Song of Ice and Fire - George R. R. Martin...","[""Rhaegar Targaryen/Original Female Character""]","[""Rhaegar Targaryen"", ""Ned Stark"", ""Sansa Star...","[""Hurt/Comfort"", ""Angst"", ""Sex"", ""Whimsical"", ...",84737721,0.000000,0.000000,0.000000,0,[],0.000000,0.196619
3,84723496,0.327024,"[""Mature""]","[""F/M""]","[""A Song of Ice and Fire & Related Fandoms"", ""...","[""Original Blackfyre Character/Ashara Dayne""]","[""Original Blackfyre Character(s)"", ""Original ...","[""Alternate Universe - Canon Divergence"", ""Hou...",84723496,0.000000,0.000000,0.000000,0,[],0.000000,0.196215
4,84761796,0.280035,"[""Not Rated""]",[],"[""Game of Thrones (TV)"", ""A Song of Ice and Fi...",[],"[""Sansa Stark"", ""Arya Stark"", ""Robb Stark"", ""J...","[""Magic"", ""shortstory"", ""Oneshot"", ""magicmirro...",84761796,0.000000,0.000000,0.000000,0,[],0.000000,0.168021
5,84722136,0.273718,"[""Mature""]","[""F/F"", ""F/M""]","[""A Song of Ice and Fire - George R. R. Martin...",[],[],[],84722136,0.000000,0.000000,0.000000,0,[],0.000000,0.164231
6,84701301,0.256271,"[""Mature""]","[""F/M""]","[""Game of Thrones (TV)""]",[],"[""Jon Snow"", ""Ghost | Jon Snow's Direwolf"", ""N...","[""Westeros (A Song of Ice and Fire)"", ""Essos (...",84701301,0.000000,0.000000,0.000000,0,[],0.000000,0.153762
7,84708881,0.186059,"[""Mature""]","[""F/M""]","[""Game of Thrones (TV)"", ""A Song of Ice and Fi...","[""Robb Stark/You""]","[""Robb Stark"", ""Catelyn Tully Stark"", ""You""]","[""I miss Robb"", ""Married Life"", ""Not even lowk...",84708881,0.000000,0.000000,0.000000,0,[],0.000000,0.111635
8,84748536,0.162172,"[""Teen And Up Audiences""]","[""F/M""]","[""Game of Thrones (TV)""]","[""Jorah Mormont/Daenerys Targaryen""]","[""Jorah Mormont"", ""Daenerys Targaryen""]",[],84748536,0.000000,0.000000,0.000000,0,[],0.000000,0.097303
9,84752771,0.159890,"[""Teen And Up Audiences""]","[""M/M""]","[""A Song of Ice and Fire - George R. R. Martin...","[""Jon Snow/Robb Stark""]","[""Robb Stark"", ""Jon Snow"", ""Grey Wind | Robb S...","[""Drunk Jon Snow"", ""Love Confessions"", ""Drunke...",84752771,0.000000,0.000000,0.000000,0,[],0.000000,0.095934
